In [1]:
!pip3 install torch=='2.3.1'
!pip3 install  accelerate=='0.27.2' peft=='0.7.1' bitsandbytes=='0.41.3.post2' #transformers=='4.36.1'
!pip3 install trl=='0.9.4'
#!pip3 install transformers=='4.35.2'
!pip3 install transformers=='4.42.3'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 1.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 89.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 74.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 9.4 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 3.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 6.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 8.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 9.3 MB/s eta 0:00:00:00:0100:01
   

In [2]:
!python -m bitsandbytes

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
++++++++++++++++++ BUG REPORT INFORMATION ++++++++++++++++++
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

++++++++++++++++++ /usr/local CUDA PATHS +++++++++++++++++++
/usr/local/nvidia/lib64/libcuda.so
/usr/local/cuda-12.1/targets/x86_64-linux/lib/libcudart.so
/usr/local/cuda-12.1/compat/libcuda.so

+++++++++++++++ WORKING DIRECTORY CUDA PATHS +++++++++++++++


++++++++++++++++++ LD_LIBRARY CUDA PATHS +++++++++++++++++++
++++++++++++ /usr/local/nvidia/lib64 CUDA PATHS ++++++++++++
/usr/local/nvidia/lib64/libcuda.so
++++++++++++++++ /opt/conda/lib CUDA PATHS +++++++++++++++++
/opt/conda/lib/python3.10/site-packages/bitsandbytes/libbitsandbytes_cuda110.so
/opt/conda/lib/python3.10/site-packages/bitsandbytes/libbitsandbytes_cuda121.so
/opt/conda/lib/python3.10/site-packages/bitsandbytes/libbitsandbytes_cuda122_nocublaslt.so
/opt/conda/lib/python3.10/site-packages/bitsandbytes/libbitsandbytes_cuda115.so
/opt/cond

In [3]:
from huggingface_hub import notebook_login

In [5]:
notebook_login()
import os
#os.environ["WANDB_PROJECT"]="openhathi_instruct_finetuning"
#os.environ['CUDA_LAUNCH_BLOCKING'] = 1
from enum import Enum
from functools import partial
import pandas as pd
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed
from datasets import load_dataset
from trl import SFTTrainer
from peft import get_peft_model, LoraConfig, TaskType
from transformers import default_data_collator, get_linear_schedule_with_warmup

seed = 42
set_seed(seed)

2024-07-08 02:32:09.461978: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-07-08 02:32:09.462075: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-07-08 02:32:09.609965: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [6]:
from datasets import load_dataset

ds = load_dataset("FinGPT/fingpt-fiqa_qa")
dataset = ds["train"].select(range(5000))
#dataset = ds["train"]
#print(ds)
dataset = dataset.train_test_split(0.1)
print(dataset)

Generating train split:   0%|          | 0/17110 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 4500
    })
    test: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 500
    })
})


In [7]:
dataset["train"][0]

{'input': 'Apartment lease renewal - is this rate increase normal?',
 'output': "I think people are missing the most obvious thing.  The yearly rate increases are just part of the landlord schtick and it is good business for them.  My grandmother owned several large apartment complexes.  She would raise rates for any resident that had been there between 1-5 years by 5-7% a year.  Even when she had vacancies and property values didn't go up.  For the following reasons: So yes it is not only normal but just part of the business.  If there are better apartments for less money I suggest you move there.  Soon those other apartments will even out and if they are better they will be much more.  So if you see a gap take advantage of it.  If you would rather stay, then simply say you will not pay the increase.  There is no use arguing about why.  The landlord will either be OK with it or say no.  Probably the biggest factors include whether you will tell other tenants (or their perception if yo

In [8]:
import torch
print(torch.cuda.is_available())

True


In [9]:
#model_name =  "mistralai/Mistral-7B-v0.1"
from transformers import BitsAndBytesConfig
model_name =  "google/gemma-2b-it"
#model_name =  "gpt2"
#compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    #device_map="auto",
    device_map={'':torch.cuda.current_device()},
    quantization_config=bnb_config
    #device_map={"":0}
)

# model.config.use_cache = False
# model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          add_eos_token=True
                                         )

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

`config.hidden_act` is ignored, you should use `config.hidden_activation` instead.
Gemma's activation function will be set to `gelu_pytorch_tanh`. Please, use
`config.hidden_activation` if you want to override this behaviour.
See https://github.com/huggingface/transformers/pull/29402 for more details.


...

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [10]:
def generate_prompt(data_point):
    # prefix_text = 'Below is an instruction that describes a task. Write a response that ' \
    #            'appropriately completes the request.\n\n'
    return{"content": f"""<start_of_turn>{data_point["instruction"]}Please generate the result as per instruction .Don't generate something on your own.Here is the input:{data_point["input"]}<end_of_turn><start_of_turn>model:{data_point["output"]}<end_of_turn>""".strip()
    }

train_dataset =  dataset["train"].map(
    generate_prompt,
    #batched=True,
    #remove_columns=dataset["train"].column_names
    #columns= ["text"]
)
eval_dataset = dataset["test"].map(
     generate_prompt,
    #batched=True,
    #remove_columns=dataset["test"].column_names
    #columns = ["text"]
)

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [11]:
from transformers import pipeline
i=0
prompt = eval_dataset[i]["content"].split("<start_of_turn>model:")[0]+"<start_of_turn>model:"
#print(prompt,"..prompt...")
# pipe = pipeline(task="text-generation",
#                         model=model,
#                         tokenizer=tokenizer,
#                         max_new_tokens = 100
#                         #temperature = 0.0,
#                       )
# result = pipe(prompt, pad_token_id=pipe.tokenizer.eos_token_id)
# print(eval_dataset[i]["content"],"*********ORIGNAL CONTENT*******")
# print(prompt,"*******PROMPT*******")
# print(result[0]["generated_text"],"******RESULT******")
# print(">>"*100)
device = "cuda:0"
encodeds = tokenizer(prompt, return_tensors="pt", add_special_tokens=True)
model_inputs = encodeds.to(device)
generated_ids = model.generate(**model_inputs, max_new_tokens=1000, do_sample=True, pad_token_id=tokenizer.eos_token_id)
# decoded = tokenizer.batch_decode(generated_ids)
decoded = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(decoded)

/opt/conda/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/opt/conda/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


Utilize your financial knowledge, give your answer or opinion to the input question or subject . Answer format is not limited.Please generate the result as per instruction .Don't generate something on your own.Here is the input:What gives non-dividend stocks value to purchasers? [duplicate]model:Non-dividend stocks provide value to purchasers by offering the potential for capital appreciation. Although they do not pay out dividends like dividend-paying stocks, companies issue new stocks to raise capital, resulting in increased supply of shares outstanding. This increased availability of shares can lead to a higher price per share in the stock market. Additionally, non-dividend stocks often have lower dividend yields compared to dividend-paying stocks. However, this disadvantage is often balanced by the potential for significantly higher capital appreciation.


In [12]:
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)
#
print(model)

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): Embedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaSdpaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): GemmaRotaryEmbedding()
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=16384, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=16384, bias=False)
          (down_proj): Linear4bit(in_features=16384, out_features=2048, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): GemmaRMSNorm()
        (post_attention_layernorm): GemmaRMSNorm()
    

In [13]:
import bitsandbytes as bnb
def find_all_linear_names(model):
  cls = bnb.nn.Linear4bit #if args.bits == 4 else (bnb.nn.Linear8bitLt if args.bits == 8 else torch.nn.Linear)
  lora_module_names = set()
  for name, module in model.named_modules():
    if isinstance(module, cls):
      names = name.split('.')
      lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names: # needed for 16-bit
      lora_module_names.remove('lm_head')
  return list(lora_module_names)
#
modules = find_all_linear_names(model)
print(modules)

['gate_proj', 'k_proj', 'q_proj', 'v_proj', 'o_proj', 'up_proj', 'down_proj']


In [14]:
peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.05,
    r=64,
    bias="none",
    target_modules=modules,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model,peft_config)

In [15]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f'trainable model parameters: {trainable_model_params}\n \
            all model parameters: {all_model_params} \n \
            percentage of trainable model parameters: {(trainable_model_params / all_model_params) * 100} %'


print(print_number_of_trainable_model_parameters(model))

trainable model parameters: 78446592
             all model parameters: 1593714688 
             percentage of trainable model parameters: 4.922248165915128 %


In [16]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side='right'
torch.cuda.empty_cache()
training_arguments = TrainingArguments(
    output_dir="logs",
    #num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4, # 4
    #optim="paged_adamw_32bit",
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    #save_steps=0,
    logging_steps=1,
    learning_rate=2e-4,
    weight_decay=0.001,
    #fp16=True,
    bf16=True,
    #max_grad_norm=0.3,
    max_steps=100,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="tensorboard",
    save_strategy="epoch"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    #eval_dataset=eval_dataset,
    peft_config=peft_config,
    dataset_text_field="content",
    tokenizer=tokenizer,
    args=training_arguments,
    packing=False,
    max_seq_length=2048,
)


/opt/conda/lib/python3.10/site-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1961: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:269: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:307: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

max_steps is given, it will override any value given in num_train_epochs


In [17]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss
1,3.399500
2,3.755400
3,3.623400
4,3.035000
5,3.114000
6,3.175600
7,3.078200
8,3.137400
9,2.920000
10,2.734400


TrainOutput(global_step=100, training_loss=2.4311807358264925, metrics={'train_runtime': 638.5291, 'train_samples_per_second': 0.626, 'train_steps_per_second': 0.157, 'total_flos': 1449603569946624.0, 'train_loss': 2.4311807358264925, 'epoch': 0.08888888888888889})

In [26]:
from transformers import pipeline
i=10
prompt = eval_dataset[i]["content"].split("<start_of_turn>model:")[0]+"<start_of_turn>model:"
#print(prompt,"..prompt...")
# pipe = pipeline(task="text-generation",
#                         model=model,
#                         tokenizer=tokenizer,
#                         max_new_tokens = 100
#                         #temperature = 0.0,
#                       )
# result = pipe(prompt, pad_token_id=pipe.tokenizer.eos_token_id)
# print(eval_dataset[i]["content"],"*********ORIGNAL CONTENT*******")
# print(prompt,"*******PROMPT*******")
# print(result[0]["generated_text"],"******RESULT******")
# print(">>"*100)
device = "cuda:0"
encodeds = tokenizer(prompt, return_tensors="pt", add_special_tokens=True)
model_inputs = encodeds.to(device)
generated_ids = model.generate(**model_inputs, max_new_tokens=75, do_sample=True, pad_token_id=tokenizer.eos_token_id)
# decoded = tokenizer.batch_decode(generated_ids)
decoded = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(decoded.split("model:")[-1])

 It depends on what type of gym membership your employer gives you on a full day basis. For example, in some cases "a day a month " or "two days a month" a normal membership is the same as a corporate membership. In these cases you really have to consider when the membership is cheaper, which is when the membership is good for most days of the month


In [27]:
decoded

'Share your insights or perspective on the financial matter presented in the input.Please generate the result as per instruction .Don\'t generate something on your own.Here is the input:How can I save money on a gym / fitness membership? New Year\'s Resolution is to get in shape - but on the cheap!model: It depends on what type of gym membership your employer gives you on a full day basis. For example, in some cases "a day a month " or "two days a month" a normal membership is the same as a corporate membership. In these cases you really have to consider when the membership is cheaper, which is when the membership is good for most days of the month'

In [28]:
i=10
eval_dataset[i]["output"]

"Find a physical activity or programme that interests you.  Memberships only have real value if you use them.  Consider learning a martial art like karate, aikido, kung fu, tai kwan do, judo, tai chi chuan. :-)   Even yoga is a good form of exercise. Many of these are offered at local community centres if you just want to try it out without worrying about the cost initially.  Use this to gauge your interest before considering more advanced clubs.  One advantage later on if you stay with it long enough - some places will compensate you for being a junior or even associate instructor. Regardless of whether this is your interest or if the gym membership is more to your liking real value is achieved if you have a good routine and interest in your physical fitness activity.  It also helps to have a workout buddy or partner.  They will help motivate you to try even when you don't feel like working out."